# Setup a FHIR Server in a new InterSystems IRIS Environment

If you have run the [docker-iris-fhir/start.sh](../../docker-iris-fhir/start.sh) script, as detailed so far, these steps have already be run for you. 

However, if you wish to create a new FHIR Server, or create a FHIR server on a clean instance of InterSystems IRIS For Health (e.g. on a cloud deployment) keep reading...

## Pre-requisities

To create a FHIR Server on a new InterSystems IRIS for Health deployment you need the following: 

- Server address (IP Address or web address)
- Superserver port (1972 is the default)
- Username and Password

For demo deployments, you can use the user `SuperUser`. If you have not changed the password, the default password is `SYS`. Please note, the password has expired by default, so you will be prompted to change it on first login through the management portal (available at  http://{SERVER}:52773/csp/sys/%25CSP.Portal.Home.zen). 

**If the password has expired, you cannot connect from Python**. 

**Please note, there may be a couple of minutes between the container starting and InterSystems IRIS for Health starting, during which time the connection will fail.**


## Things to define

Along with this information, you also need to define (i.e. create names) for the following: 

- **Namespace name** (the namespace will be created below).
    - A namespace is an organisational unit in IRIS which includes data and code. For a hackathon project you can probably just stick to a single namespace with a simple name.
    - The rest of the tutorial uses `FHIRSERVER`, you can use the same in your deployment.


- **FHIR Server endpoint**
    - This is the address at which your FHIR server will be available.
    - It needs to start with a `/` and be a valid URL path.
    - Again, the tutorial uses `/fhir/r4`, you can do the same here or use something different.





## Code

To create a new FHIR Server, you can use the following: 

In [1]:
import iris


## Set credentials here  
HOST = "localhost" # could ba an IP address, e.g. 192.12.22.142
PORT = 52773 # Default super server port 
USER = "superuser" # Default Username 
PASSWORD = "SYS" # Default password 



def connect(namespace):
    """
    Connect to IRIS with Python Native API connection 
    using the global connection parameters (Defined above)
    """
    conn = iris.connect(HOST, PORT, namespace, USER, PASSWORD)
    return conn, iris.createIRIS(conn)


def create_fhir_server(namespace, endpoint, overwrite=False):
    """
    Create a FHIR server in a specified namespace and a defined endpoint 
    """
    
    conn, irispy = connect("HSLIB")
    try:
        ns_exists = irispy.classMethodBoolean("%SYS.Namespace", "Exists", namespace)
        if not ns_exists:
            irispy.classMethodVoid("HS.Util.Installer.Foundation", "Install", namespace)
    finally:
        conn.close()

    conn, irispy = connect(namespace)
    try:
        exists = irispy.classMethodBoolean("HS.FHIRServer.ServiceInstance", "ExistsByUrl", endpoint)

        if exists and not overwrite:
            print("FHIR server already exists, skipping...")
            return

        if exists and overwrite:
            print("FHIR server exists, overwriting...")
            irispy.classMethodVoid("HS.FHIRServer.Installer", "UninstallInstance", endpoint)

        strategy_class = "HS.FHIRServer.Storage.JsonAdvSQL.InteractionsStrategy"
        metadata = "hl7.fhir.r4.core@4.0.1"

        irispy.classMethodVoid("HS.FHIRServer.Installer", "InstallNamespace")
        irispy.classMethodVoid("HS.FHIRServer.Installer", "InstallInstance", endpoint, strategy_class, metadata)
    finally:
        conn.close()


create_fhir_server("FHIRSERVER", "/fhir/r4")



HS.FHIRServer.Installer:InstallNamespace Scheduled FHIR Purge Expired Search Results TaskMeta: 4840@HS.FHIRMeta.Impl.Impl, repo: 2, svc: 1
Generating Tables

Saving: HSFHIR.X0002.R.Rsrc
Saving: HSFHIR.X0002.V.RsrcVer
Compiling HSFHIR.X0002.R.*... 
Compilation started on 10/13/2026 15:37:58 with qualifiers 'e'
Compiling class HSFHIR.X0002.R.Rsrc
Compiling table HSFHIR_X0002_R.Rsrc
Compiling routine HSFHIR.X0002.R.Rsrc.1
Compilation finished successfully in 0.100s.

Compilation started on 10/13/2026 15:37:58 with qualifiers 'e'
Compiling class HSFHIR.X0002.V.RsrcVer
Compiling table HSFHIR_X0002_V.RsrcVer
Compiling routine HSFHIR.X0002.V.RsrcVer.1
Compilation finished successfully in 0.086s.

Generating HSFHIR.X0002.S.*... 
PackageDefinition for 'HSFHIR.X0002.S' created
Saving: HSFHIR.X0002.S.AccountCompartments
Saving: HSFHIR.X0002.S.Account.identifier
Saving: HSFHIR.X0002.S.Account.patient
Saving: HSFHIR.X0002.S.Account.subject
Saving: HSFHIR.X0002.S.Account.type
Saving: HSFHIR.X0002.S

This code above should take a while to run but will create the FHIR server. 

The address will depend on the web port. If you are using the community edition of InterSystems IRIS For Health (which you probably are), the default web port is `52773`. The FHIR address is as follows: 

```
http://<SERVER>:<WEB PORT><ENDPOINT>
```
This translates to: 

http://123.45.67.890:52783/fhir/r4 

You can check that the FHIR Server is functioning by going to http://123.45.67.890:52783/fhir/r4/metadata, which should download a FHIR Capability statement file if it is functioning. 

# Load FHIR data via HTTP

If you have a directory full of FHIR data files (JSON), you can add them to the FHIR Server with the following:

In [3]:
import base64
import urllib.request
from pathlib import Path

# Again, we need credentials

HOST = "localhost" # could ba an IP address, e.g. 192.12.22.142
WEB_PORT = 32783 # Default web server port 
USER = "superuser" # Default Username 
PASSWORD = "SYS" # Default password 

# Created above
FHIR_END_POINT = "/fhir/r4"

def load_fhir_data_http(data_path:str):
        # Bundles sorted so hospital/practitioner info loads before patient bundles that reference them

        data_path = Path(data_path)

        ## Finds the files and ensures the hospital/practitioner info is sent first
        files = sorted(
            data_path.glob("*.json"),
            key=lambda p: (not p.name.startswith(("hospitalInformation", "practitionerInformation")), p.name),
        )

    
        if not files:
            raise FileNotFoundError(f"No FHIR bundles found in {data_path}")

        # Create full FHIR Server URL
        base_url = f"http://{HOST}:{WEB_PORT}{FHIR_END_POINT}"

        # Create basic authentication 
        auth = base64.b64encode(f"{USER}:{PASSWORD}".encode()).decode()


        # for each file 
        for i, path in enumerate(files, 1):
            
            print(f"[{i}/{len(files)}] POST {path.name} -> {base_url}")

            # POST request with file data
            req = urllib.request.Request(
                base_url,
                data=path.read_bytes(),
                method="POST",
                headers={
                    "Content-Type": "application/fhir+json",
                    "Accept": "application/fhir+json",
                    "Authorization": f"Basic {auth}",
                },
            )
            # urlopen raises HTTPError on 4xx/5xx
            with urllib.request.urlopen(req, timeout=600) as resp:
                print(f"    HTTP {resp.status}")

load_fhir_data_http("./data/fhir")

FileNotFoundError: No FHIR bundles found in data\fhir

# Bulk Load

If you have large datasets, particularly for files which are very large, POST requests may lead to time out issues or slow loading. For these, you are better off the InterSystems FHIR server data loader method, which can be done with the Python Native API. 

However, bulk loading requires the data to be on the same file system as InterSystems IRIS. Therefore, for deployments (or for a local docker container), the first step is to copy the files to the file systems. 

For example: 


In [ ]:
!scp -r fhirdata irisowner@123-45-67-890:/tmp/fhirdata

In [ ]:
!docker cp fhirdata iris-fhir:/tmp/fhirdata

After this, you can load the data into InterSystems IRIS using the `HS.FHIRServer.Tools.DataLoader` class:

In [ ]:
HOST = "localhost" # could ba an IP address, e.g. 192.12.22.142
PORT = 1972 # Default super server port 
USER = "superuser" # Default Username 
PASSWORD = "SYS" # Default password 
NAMESPACE = "FHIRSERVER" # Created above

DATA_PATH = "/tmp/fhirdata"

conn_args = {"hostname": HOST, "port":PORT, "namespace":NAMESPACE, "username": USERNAME, "password":PASSWORD}

conn = iris.connect(**conn_args)

irispy = iris.createIRIS(conn) 

status = irispy.classMethodString("HS.FHIRServer.Tools.DataLoader", # IRIS Class 
                                   "SubmitResourceFiles", # Method name
                                  DATA_PATH, # Path to FHIR bundle 
                                   "FHIRServer", # Type of load (Ignore)
                                   "/fhir/r4", # 
                                   1 # Display progress
                                 )

print(status)
conn.close()
time_taken = time.time() - st
print(time_taken)

